# W&B Skills + HuggingFace 실험 환경 설정

**코딩 에이전트(Claude Code)와 W&B Skills를 연동하여 ML 실험을 효율적으로 관리하는 노트북**

### 이 노트북의 목적
- W&B Skills 설치 → 코딩 에이전트가 실험 추적/평가를 자동 처리
- HuggingFace 모델/데이터셋을 미리 정의 → 에이전트 토큰 절약
- 에이전트에게 매번 설명 없이 모델 ID만 참조하면 바로 실험 가능

---

## 1단계: 기본 환경 및 인증 설정

In [ ]:
import os

# ========================================
# 인증 토큰 설정 (본인 토큰으로 교체)
# ========================================
WANDB_API_KEY = "YOUR_WANDB_API_KEY"          # https://wandb.ai/authorize 에서 발급
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"            # https://huggingface.co/settings/tokens 에서 발급
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"             # https://github.com/settings/tokens 에서 발급

# 환경변수 등록
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GH_TOKEN'] = GITHUB_TOKEN

# Git 설정
!git config --global user.name "umyunsang"
!git config --global user.email "dbstkd5865@gmail.com"

# .bashrc 영구 등록
with open(os.path.expanduser('~/.bashrc'), 'a') as f:
    f.write(f"\n# === API Keys ===\n")
    f.write(f"export WANDB_API_KEY={WANDB_API_KEY}\n")
    f.write(f"export HF_TOKEN={HF_TOKEN}\n")
    f.write(f"export GITHUB_TOKEN={GITHUB_TOKEN}\n")
    f.write(f"export GH_TOKEN={GITHUB_TOKEN}\n")

print("\u2705 인증 환경변수 설정 완료")

## 2단계: ML 패키지 설치

In [ ]:
# W&B + Weave + HuggingFace + 학습 관련 패키지 일괄 설치
!pip install -q \
    wandb \
    weave \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    autoawq \
    huggingface_hub \
    torch torchvision torchaudio \
    sentencepiece \
    protobuf

print("\u2705 패키지 설치 완료")

# 버전 확인
import wandb, transformers, peft, trl, torch
print(f"wandb: {wandb.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")
print(f"torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")

## 3단계: W&B 로그인 및 프로젝트 초기화

In [ ]:
import wandb

# W&B 로그인
wandb.login(key=os.environ.get('WANDB_API_KEY'))

print("\u2705 W&B 로그인 성공")
print(f"   대시보드: https://wandb.ai")

## 4단계: W&B Skills 설치 (코딩 에이전트 연동)

W&B Skills는 코딩 에이전트가 실험 추적, 트레이싱, 평가를 자동으로 수행할 수 있게 해주는 스킬 패키지입니다.

### W&B Skills 제공 기능
| 기능 | 설명 | 성공률 (Claude Code) |
|------|------|---------------------|
| **Experiment Tracking** | 학습 loss, metrics 자동 로깅 | ~91% |
| **Tracing** | 에이전트 동작 흐름 기록 (Weave) | ~91% |
| **Evaluations** | 모델 성능 평가 자동화 | ~91% |

In [ ]:
# Node.js 설치 (npx skills 실행에 필요)
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
!sudo apt-get install -y -qq nodejs

# W&B Skills 설치 (Claude Code 글로벌)
!npx skills add wandb/skills --global

print("\n\u2705 W&B Skills 설치 완료")
print("   코딩 에이전트가 W&B 실험 추적/평가를 자동으로 처리합니다.")

## 5단계: HuggingFace 로그인

In [ ]:
from huggingface_hub import login

login(token=os.environ.get('HF_TOKEN'))

print("\u2705 HuggingFace 로그인 성공")
print("   프로필: https://huggingface.co/umyunsang")

---

## 6단계: 내 HuggingFace 모델 레지스트리

코딩 에이전트에게 전달할 모델 정보를 미리 정의합니다.  
에이전트가 이 셀을 참조하면 모델 ID, 용도, 설정을 즉시 파악 → **토큰 절약**

### 내 모델 파이프라인 (EXAONE 8B 민원 분류)

```
Base Model (EXAONE 3.5 8B)
    │
    ├── [1] QLoRA Fine-tuning
    │   └── umyunsang/civil-complaint-exaone-lora        (LoRA 어댑터)
    │
    ├── [2] Merge (LoRA + Base)
    │   └── umyunsang/civil-complaint-exaone-merged      (Full Weight, 8B)
    │
    └── [3] AWQ Quantization
        └── umyunsang/civil-complaint-exaone-awq         (4bit 양자화, 배포용)
```

In [ ]:
# =============================================
# 내 HuggingFace 모델 레지스트리
# 코딩 에이전트가 이 변수를 참조하여 실험 수행
# =============================================

MY_MODELS = {
    # === 민원 분류 모델 (EXAONE 8B 기반) ===
    "lora_adapter": {
        "model_id": "umyunsang/civil-complaint-exaone-lora",
        "base_model": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        "type": "LoRA Adapter",
        "task": "민원 분류 (Text Classification)",
        "method": "QLoRA (4bit)",
        "usage": "PEFT로 로드하여 추가 학습 또는 평가",
    },
    "merged_full": {
        "model_id": "umyunsang/civil-complaint-exaone-merged",
        "base_model": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        "type": "Full Merged Model (FP16)",
        "task": "민원 분류 (Text Classification)",
        "method": "LoRA merged into base",
        "vram": "~16GB (FP16)",
        "usage": "AWQ/GPTQ 양자화 소스, 벤치마크 기준 모델",
    },
    "awq_quantized": {
        "model_id": "umyunsang/civil-complaint-exaone-awq",
        "base_model": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        "type": "AWQ 4bit Quantized",
        "task": "민원 분류 (Text Classification)",
        "method": "AWQ (4bit, group_size=128)",
        "vram": "~5GB",
        "usage": "온디바이스 배포용, vLLM/TGI 서빙",
    },

    # === 기타 ===
    "comfyui": {
        "model_id": "umyunsang/comfyui-models",
        "type": "Image-to-Image",
        "task": "이미지 생성",
        "usage": "ComfyUI 워크플로우용 모델",
    },
}

# === 베이스 모델 (파인튜닝 후보) ===
BASE_MODELS = {
    "exaone_8b": {
        "model_id": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        "params": "7.8B",
        "vram_qlora": "~12GB",
        "vram_full": "~16GB",
        "context": "32K",
        "korean": True,
        "note": "LG AI 연구원, 한국어 최강",
    },
    "exaone_2.4b": {
        "model_id": "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
        "params": "2.4B",
        "vram_qlora": "~5GB",
        "vram_full": "~6GB",
        "context": "32K",
        "korean": True,
        "note": "경량 온디바이스용, T4에서도 학습 가능",
    },
    "qwen2.5_7b": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "params": "7B",
        "vram_qlora": "~10GB",
        "vram_full": "~14GB",
        "context": "128K",
        "korean": True,
        "note": "긴 컨텍스트, 다국어 강점",
    },
    "gemma2_9b": {
        "model_id": "google/gemma-2-9b-it",
        "params": "9B",
        "vram_qlora": "~12GB",
        "vram_full": "~18GB",
        "context": "8K",
        "korean": False,
        "note": "Google, 추론 성능 우수",
    },
    "llama3.1_8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "params": "8B",
        "vram_qlora": "~10GB",
        "vram_full": "~16GB",
        "context": "128K",
        "korean": False,
        "note": "Meta, 범용 성능 최상위",
    },
}

# 출력
print("=" * 60)
print("\U0001f4e6 내 HuggingFace 모델")
print("=" * 60)
for key, model in MY_MODELS.items():
    print(f"\n  [{key}]")
    print(f"    ID:   {model['model_id']}")
    print(f"    타입: {model['type']}")
    print(f"    용도: {model['usage']}")

print("\n" + "=" * 60)
print("\U0001f3af 파인튜닝 후보 베이스 모델")
print("=" * 60)
for key, model in BASE_MODELS.items():
    kr = "\u2705" if model.get('korean') else "\u274c"
    print(f"\n  [{key}] {model['model_id']}")
    print(f"    {model['params']} | QLoRA: {model['vram_qlora']} | 한국어: {kr} | {model['note']}")

## 7단계: M3 마일스톤 목표 및 실험 설정

GitHub Milestone: [M3: 고도화 및 최적화](https://github.com/GovOn-Org/GovOn/milestone/3)  
기간: Week 9-12 (마감: 2026-05-22)

### M3 핵심 KPI (미달 → 목표)

| KPI | 현재값 (M2) | 목표 | 갭 | 관련 이슈 |
|-----|------------|------|-----|-----------|
| **BLEU** | 17.32 | >= 30 | -12.68 | #68 |
| **ROUGE-L** | 18.28 | >= 40 | -21.72 | #68 |
| **BERTScore F1** | 46.05 | >= 55 (중간) | -8.95 | #68 |
| **p50 Latency** | ~2.4s | < 2s (AWQ: <1s) | -0.4s | #69 |
| **분류 정확도** | 90% (EXAONE) | >= 95% (KR-ELECTRA) | -5% | #55 |
| **Recall@5 (FAISS)** | - | >= 80% | 미구현 | #53 |

### M3 Open Issues (11개)

**모델 최적화 (4개)**
- #67 QLoRA 하이퍼파라미터 최적화 (Rank/LR/Epoch 탐색)
- #68 답변 생성 품질 고도화 (BLEU/ROUGE-L 목표 달성)
- #69 추론 속도 추가 최적화 (Avg 2.43s → p50 < 2s)
- #70 학습 데이터 증강 및 테스트셋 품질 개선

**시스템 구축 (4개)**
- #53 FAISS 벡터 검색 시스템 구축
- #54 RAG 파이프라인 통합 구현
- #55 KR-ELECTRA 기반 전용 분류기 구축
- #56 FastAPI 백엔드 서버 구축 및 API 통합

**프론트엔드 & 배포 (2개)**
- #57 Figma MCP 기반 프론트엔드 고도화
- #58 Docker 컨테이너화 및 폐쇄망 배포

**통합 (1개)**
- #27 RAG 통합, FAISS 벡터 검색 및 전용 분류기 구축

In [ ]:
# =============================================
# 실험 설정 템플릿
# 코딩 에이전트에게 이 config를 참조하라고 지시하면 끝
# =============================================

EXPERIMENT_CONFIG = {
    # --- 프로젝트 ---
    "project": "GovOn-Org/GovOn",
    "milestone": "M3: 고도화 및 최적화",
    "milestone_url": "https://github.com/GovOn-Org/GovOn/milestone/3",
    "due_date": "2026-05-22",

    # --- W&B 프로젝트 ---
    "wandb_project": "civil-complaint-classification",
    "wandb_entity": None,

    # --- 내 HuggingFace 모델 ---
    "base_model": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    "my_lora": "umyunsang/civil-complaint-exaone-lora",
    "my_merged": "umyunsang/civil-complaint-exaone-merged",
    "my_awq": "umyunsang/civil-complaint-exaone-awq",

    # --- M3 KPI 목표 ---
    "kpi_targets": {
        "bleu": {"current": 17.32, "target": 30, "issue": "#68"},
        "rouge_l": {"current": 18.28, "target": 40, "issue": "#68"},
        "bertscore_f1": {"current": 46.05, "target": 55, "issue": "#68"},
        "p50_latency_sec": {"current": 2.4, "target": 2.0, "awq_target": 1.0, "issue": "#69"},
        "classification_accuracy": {"current": 0.90, "target": 0.95, "issue": "#55"},
        "faiss_recall_at_5": {"current": None, "target": 0.80, "issue": "#53"},
    },

    # --- M3 Baseline (EXP-001) ---
    "baseline": {
        "eval_loss": 1.0179,
        "perplexity": 3.1957,
        "bleu": 17.32,
        "rouge_l": 18.28,
        "bertscore_f1": 46.05,
        "avg_latency": 2.43,
        "throughput_tok_s": 13.9,
    },

    # --- QLoRA 하이퍼파라미터 (Baseline) ---
    "qlora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },

    # --- QLoRA 실험 매트릭스 (#67) ---
    "experiments": {
        "EXP-002a": {"r": 8,  "lora_alpha": 16,  "note": "경량화 검증"},
        "EXP-002b": {"r": 32, "lora_alpha": 64,  "note": "성능 상한선 탐색"},
        "EXP-002c": {"r": 64, "lora_alpha": 128, "note": "최대 성능 (VRAM 허용 시)"},
        "EXP-003a": {"lr": 1e-4, "scheduler": "cosine", "note": "안정적 수렴"},
        "EXP-003b": {"lr": 5e-5, "scheduler": "cosine", "note": "과적합 방지"},
        "EXP-003c": {"lr": 2e-4, "scheduler": "linear", "note": "스케줄러 비교"},
        "EXP-004a": {"epochs": 2, "note": "추가 학습 효과"},
        "EXP-004b": {"epochs": 3, "note": "완전 학습"},
    },

    # --- 학습 설정 ---
    "training": {
        "num_train_epochs": 3,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.1,
        "max_seq_length": 2048,
        "fp16": False,
        "bf16": True,
        "logging_steps": 10,
        "save_strategy": "epoch",
        "eval_strategy": "epoch",
        "report_to": "wandb",
    },

    # --- AWQ 양자화 설정 ---
    "awq": {
        "quant_method": "awq",
        "bits": 4,
        "group_size": 128,
        "zero_point": True,
        "version": "gemm",
    },

    # --- 시스템 구성 요소 ---
    "system_components": {
        "vector_search": {
            "embedding_model": "intfloat/multilingual-e5-large",
            "engine": "FAISS",
            "data_count": 10000,
            "target_recall_at_5": 0.80,
            "issue": "#53",
        },
        "classifier": {
            "model": "snunlp/KR-ELECTRA-discriminator",
            "target_accuracy": 0.95,
            "target_latency_ms": 100,
            "issue": "#55",
        },
        "rag_pipeline": {
            "flow": "Query -> Embedding -> FAISS -> Prompt Augmentation -> EXAONE Generation",
            "context_window": "32K",
            "target_bertscore_improvement": "10%+",
            "issue": "#54",
        },
        "backend": {
            "framework": "FastAPI",
            "endpoints": ["/generate", "/search", "/classify"],
            "target_p95_latency": "5s",
            "issue": "#56",
        },
        "frontend": {
            "framework": "React/Next.js",
            "design": "Figma MCP",
            "issue": "#57",
        },
        "deployment": {
            "method": "Docker (docker-compose)",
            "gpu": "NVIDIA Container Toolkit",
            "offline": True,
            "issue": "#58",
        },
    },

    # --- HuggingFace 업로드 ---
    "hub": {
        "push_to_hub": True,
        "hub_model_id_prefix": "umyunsang/civil-complaint",
    },
}

import json
print("\u2705 M3 실험 설정 템플릿 로드 완료")
print(f"\n[M3 KPI 목표]")
for kpi, v in EXPERIMENT_CONFIG['kpi_targets'].items():
    current = v['current'] if v['current'] is not None else "미측정"
    print(f"  {kpi}: {current} -> {v['target']} ({v['issue']})")
print(f"\n[실험 매트릭스] {len(EXPERIMENT_CONFIG['experiments'])}개 실험 정의됨")
for exp_id, exp in EXPERIMENT_CONFIG['experiments'].items():
    print(f"  {exp_id}: {exp['note']}")

## 8단계: W&B + Weave 연동 테스트

In [ ]:
import wandb
import weave

# ========================================
# W&B 팀(entity) 설정
# ========================================
WANDB_ENTITY = "umyun3"

# W&B 실험 추적 테스트
run = wandb.init(
    project=EXPERIMENT_CONFIG['wandb_project'],
    entity=WANDB_ENTITY,
    name="setup-test",
    tags=["test", "setup-validation"],
    config=EXPERIMENT_CONFIG,
)

# 테스트 메트릭 로깅
wandb.log({"test_metric": 1.0, "setup": "success"})
wandb.finish()

print("\u2705 W&B 실험 추적 테스트 성공")

# Weave 트레이싱 테스트
weave.init(f"{WANDB_ENTITY}/{EXPERIMENT_CONFIG['wandb_project']}")

@weave.op()
def classify_complaint(text: str) -> str:
    """민원 분류 함수 (테스트용)"""
    return "교통/도로"

result = classify_complaint("도로에 포트홀이 있습니다")
print(f"\u2705 Weave 트레이싱 테스트 성공: {result}")
print(f"   대시보드에서 추적 결과를 확인하세요.")

# EXPERIMENT_CONFIG에 entity 업데이트
EXPERIMENT_CONFIG['wandb_entity'] = WANDB_ENTITY
print(f"\n\u2705 W&B entity: {WANDB_ENTITY}")

---

## 9단계: CLAUDE.md 프로젝트 컨텍스트 생성

코딩 에이전트가 프로젝트를 이해하도록 컨텍스트 파일을 생성합니다.  
이 파일이 있으면 에이전트가 **매번 질문 없이** 바로 작업 가능 → **토큰 대폭 절약**

In [ ]:
import os
import json

# 프로젝트 디렉토리 설정
project_dir = os.path.expanduser('~/civil-complaint-project')
os.makedirs(project_dir, exist_ok=True)

# CLAUDE.md 생성 - 에이전트가 자동으로 읽는 프로젝트 컨텍스트
claude_md = """# Project: GovOn - 민원 분류 온디바이스 AI

## 프로젝트 개요
시민 민원 텍스트를 자동 분류하고 답변을 생성하는 온디바이스 AI 시스템.
GitHub: https://github.com/GovOn-Org/GovOn

## 현재 마일스톤: M3 고도화 및 최적화
- Milestone: https://github.com/GovOn-Org/GovOn/milestone/3
- 기간: Week 9-12 (마감: 2026-05-22)

## M3 핵심 KPI 및 목표
| KPI | 현재값 | 목표 | 관련 이슈 |
|-----|--------|------|-----------|
| BLEU | 17.32 | >= 30 | #68 |
| ROUGE-L | 18.28 | >= 40 | #68 |
| BERTScore F1 | 46.05 | >= 55 | #68 |
| p50 Latency | 2.4s | < 2s (AWQ: <1s) | #69 |
| 분류 정확도 | 90% | >= 95% | #55 |
| FAISS Recall@5 | 미구현 | >= 80% | #53 |

## M3 Baseline (EXP-001)
- eval_loss: 1.0179, perplexity: 3.1957
- BLEU: 17.32, ROUGE-L: 18.28, BERTScore F1: 46.05
- avg_latency: 2.43s, throughput: 13.9 tok/s

## M3 Open Issues
### 모델 최적화
- #67 QLoRA 하이퍼파라미터 최적화 (Rank/LR/Epoch 탐색)
  - EXP-002a: r=8, EXP-002b: r=32, EXP-002c: r=64
  - EXP-003a: lr=1e-4, EXP-003b: lr=5e-5, EXP-003c: lr=2e-4(linear)
  - EXP-004a: 2 epochs, EXP-004b: 3 epochs
- #68 답변 생성 품질 고도화 (BLEU/ROUGE-L)
  - 전략: 데이터 품질 개선, 프롬프트 엔지니어링, 디코딩 최적화, 평가 방법론 개선
- #69 추론 속도 최적화 (p50 < 2s)
  - 전략: vLLM 파라미터 튜닝, 생성 파라미터 최적화, Speculative Decoding
- #70 학습 데이터 증강 및 테스트셋 품질 개선
  - 카테고리별 균형 테스트셋 300건+, PII 마스킹 개선

### 시스템 구축
- #53 FAISS 벡터 검색 (intfloat/multilingual-e5-large, Recall@5 >= 80%)
- #54 RAG 파이프라인 (Query→Embedding→FAISS→Prompt Augmentation→Generation)
- #55 KR-ELECTRA 전용 분류기 (snunlp/KR-ELECTRA-discriminator, 정확도 >= 95%, < 100ms)
- #56 FastAPI 백엔드 (/generate, /search, /classify)

### 프론트엔드 & 배포
- #57 Figma MCP 기반 React/Next.js 프론트엔드
- #58 Docker 컨테이너화 및 폐쇄망 배포

## 내 HuggingFace 모델
- LoRA 어댑터: `umyunsang/civil-complaint-exaone-lora`
- Merged 모델: `umyunsang/civil-complaint-exaone-merged`
- AWQ 양자화:  `umyunsang/civil-complaint-exaone-awq`
- 베이스 모델:  `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`

## 시스템 구성 모델
- 임베딩: `intfloat/multilingual-e5-large` (벡터 검색용)
- 분류기: `snunlp/KR-ELECTRA-discriminator` (경량 민원 분류)
- 생성: `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` (답변 생성, vLLM 서빙)

## 실험 추적
- W&B 프로젝트: `civil-complaint-classification`
- 모든 학습은 `report_to="wandb"`로 자동 로깅
- Weave로 추론 트레이싱 활성화
- experiment_config.json에 전체 하이퍼파라미터 정의됨

## QLoRA 기본 설정 (Baseline)
- r=16, alpha=32, dropout=0.05
- target: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- epochs: 3, batch: 4x4, lr: 2e-4, cosine, bf16, max_seq: 2048

## AWQ 양자화 설정
- bits: 4, group_size: 128, zero_point: True

## 규칙
- 커밋 메시지는 한글로 작성
- Conventional Commits 형식 (feat, fix, docs 등)
- Co-Authored-By 절대 금지
- HuggingFace 업로드 시 prefix: `umyunsang/civil-complaint`
- 실험 결과는 반드시 W&B에 로깅
"""

with open(os.path.join(project_dir, 'CLAUDE.md'), 'w') as f:
    f.write(claude_md)

# 실험 설정도 JSON으로 저장 (에이전트가 직접 로드 가능)
with open(os.path.join(project_dir, 'experiment_config.json'), 'w') as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2, ensure_ascii=False)

print(f"\u2705 프로젝트 디렉토리: {project_dir}")
print(f"\u2705 CLAUDE.md 생성 완료 (M3 마일스톤 목표 포함)")
print(f"\u2705 experiment_config.json 생성 완료 (KPI + 실험 매트릭스 + 시스템 구성)")
print(f"\n\U0001f4a1 에이전트에게 이렇게 지시하면 됩니다:")
print(f'   \"experiment_config.json을 참조하여 #67 QLoRA 하이퍼파라미터 최적화를 수행해줘\"')

---

## 10단계: 전체 검증

In [ ]:
import shutil

print("=" * 55)
print("\U0001f50d 전체 설정 검증")
print("=" * 55)

checks = []

# 1. GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"1. GPU: \u2705 {gpu_name} ({gpu_mem:.0f}GB)")
    checks.append(True)
else:
    print(f"1. GPU: \u274c CUDA 사용 불가")
    checks.append(False)

# 2. W&B
wandb_key = os.environ.get('WANDB_API_KEY', '')
if wandb_key and wandb_key != 'YOUR_WANDB_API_KEY':
    print(f"2. W&B: \u2705 API 키 설정됨")
    checks.append(True)
else:
    print(f"2. W&B: \u26a0\ufe0f API 키 미설정 (1단계에서 설정 필요)")
    checks.append(False)

# 3. HuggingFace
hf_key = os.environ.get('HF_TOKEN', '')
if hf_key and hf_key != 'YOUR_HUGGINGFACE_TOKEN':
    print(f"3. HuggingFace: \u2705 토큰 설정됨")
    checks.append(True)
else:
    print(f"3. HuggingFace: \u26a0\ufe0f 토큰 미설정 (1단계에서 설정 필요)")
    checks.append(False)

# 4. GitHub
gh_key = os.environ.get('GITHUB_TOKEN', '')
if gh_key:
    print(f"4. GitHub: \u2705 토큰 설정됨")
    checks.append(True)
else:
    print(f"4. GitHub: \u274c 토큰 미설정")
    checks.append(False)

# 5. CLAUDE.md
claude_md_path = os.path.expanduser('~/civil-complaint-project/CLAUDE.md')
if os.path.exists(claude_md_path):
    print(f"5. CLAUDE.md: \u2705 존재")
    checks.append(True)
else:
    print(f"5. CLAUDE.md: \u274c 없음")
    checks.append(False)

# 6. experiment_config.json
config_path = os.path.expanduser('~/civil-complaint-project/experiment_config.json')
if os.path.exists(config_path):
    print(f"6. experiment_config.json: \u2705 존재")
    checks.append(True)
else:
    print(f"6. experiment_config.json: \u274c 없음")
    checks.append(False)

# 7. 패키지
try:
    import wandb, weave, transformers, peft, trl
    print(f"7. ML 패키지: \u2705 wandb, weave, transformers, peft, trl")
    checks.append(True)
except ImportError as e:
    print(f"7. ML 패키지: \u274c {e}")
    checks.append(False)

print("\n" + "=" * 55)
if all(checks):
    print("\U0001f389 모든 설정 완료! 실험을 시작할 수 있습니다.")
else:
    print(f"\u26a0\ufe0f {sum(checks)}/{len(checks)} 항목 통과. 위 항목을 확인하세요.")

---

## 사용 예시: 코딩 에이전트에게 지시하는 방법

이슈 번호로 지시하면 에이전트가 CLAUDE.md + experiment_config.json을 참조하여 바로 작업합니다.

### #67 QLoRA 하이퍼파라미터 최적화
```
#67 이슈 작업을 시작해. experiment_config.json의 experiments 매트릭스에 정의된
EXP-002a~002c (Rank 변화) 실험을 순차적으로 실행하고 W&B에 로깅해.
각 실험의 eval_loss, BLEU, ROUGE-L을 baseline과 비교해줘.
```

### #68 답변 생성 품질 고도화
```
#68 이슈 작업. 현재 BLEU 17.32 → 목표 30 달성을 위해
repetition_penalty, temperature, max_new_tokens 조합을 실험해.
Weave로 추론 트레이싱하고 W&B에 결과 로깅해.
```

### #69 추론 속도 최적화
```
#69 이슈 작업. umyunsang/civil-complaint-exaone-awq 모델로
vLLM 서빙 파라미터(max_model_len, gpu_memory_utilization)를 튜닝해서
p50 latency < 2s를 달성해. 10회 반복 벤치마크 필수.
```

### #55 KR-ELECTRA 분류기 구축
```
#55 이슈 작업. snunlp/KR-ELECTRA-discriminator를 민원 6개 카테고리로
파인튜닝해줘. 목표 정확도 95%+, 추론 100ms 이내. W&B에 로깅해.
```

### #53 FAISS 벡터 검색
```
#53 이슈 작업. intfloat/multilingual-e5-large로 민원 데이터 10,000건을
임베딩하고 FAISS 인덱스를 빌드해. Recall@5 >= 80% 검증해줘.
```

### #54 RAG 파이프라인
```
#54 이슈 작업. #53 FAISS 검색 결과를 EXAONE 프롬프트에 주입하는
RAG 파이프라인을 구현해. BERTScore 10%+ 향상 목표.
```

### #56 FastAPI 백엔드
```
#56 이슈 작업. vLLM + FAISS + KR-ELECTRA를 통합하는
FastAPI 서버를 구현해. /generate, /search, /classify 엔드포인트.
```

### #70 데이터 증강
```
#70 이슈 작업. 카테고리별 균형 잡힌 테스트셋 300건+을 구성하고
데이터 분포 분석 리포트를 작성해.
```